## Locations

Paths are relative to the data directory given by the environment variable `M26_DATA_ROOT` (see the README); figure panels are written to `M26_RESULTS_ROOT`.

In [ ]:
# Locations (see README): M26_DATA_ROOT is required; M26_RESULTS_ROOT defaults to <repo>/results
import os
from pathlib import Path

REPO_ROOT = os.environ.get("M26_REPO_ROOT") or str(
    next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "environment" / "downstr.yml").is_file())
)
os.environ["M26_REPO_ROOT"] = REPO_ROOT
DATA_ROOT = os.environ["M26_DATA_ROOT"]
RESULTS_ROOT = os.environ.get("M26_RESULTS_ROOT", f"{REPO_ROOT}/results")
# Run in the directory this notebook originally ran in, so that relative paths resolve as they did
WORK_DIR = Path(DATA_ROOT, "notebooks/3UTR/Wildtype_mice/2026")
WORK_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(WORK_DIR)

In [ ]:
import os
from collections import defaultdict

import numpy as np

import pandas as pd
import scanpy as sc

import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

from scipy.sparse import issparse, diags
from scipy.stats import spearmanr, rankdata, entropy as shannon_entropy
from statsmodels.stats.multitest import multipletests

# Load data


In [ ]:
BASEPATH = f"{DATA_ROOT}/3utr/wildtypes_analysis/2026_analyses/utrcalling"
STOREPATH = os.path.join(BASEPATH, "correlation_analyses_proportions")
os.makedirs(STOREPATH, exist_ok=True)

In [ ]:
adata = sc.read_h5ad(
    os.path.join(
        BASEPATH,
        "UTR_count_table_genelevel_bw12_above10cts_apa_named_adata_harmony_lineage.h5ad",
    )
)

adata

# Check UMAP


In [ ]:
sc.pl.umap(
    adata,
    color=[
        "leiden",
        "phase",
        "batch_number",
        "dpt_pseudotime",
        "n_genes_by_counts",
        "celltype",
    ],
    wspace=0.4,
    ncols=2,
)

# Find genes changing upon pseudotime


In [ ]:
adata.obs.dpt_pseudotime.value_counts(dropna=False)

In [ ]:
display(adata)
display(adata[~adata.obs["dpt_pseudotime"].isna()])

In [ ]:
adata = adata[~adata.obs["dpt_pseudotime"].isna()]
adata

In [ ]:
# The harmony lineage file (.X) contains z-scored expression (normalize -> log1p -> scale),
# not raw counts. Load the original count matrix for graph smoothing.
adata_raw = sc.read_h5ad(
    os.path.join(BASEPATH, "UTR_count_table_genelevel_bw12_above10cts_apa_named.h5ad")
)
adata_raw = adata_raw[adata.obs_names].copy()

X_raw = adata_raw.X
if issparse(X_raw):
    X_raw = X_raw.toarray().astype(np.float64)
else:
    X_raw = np.asarray(X_raw, dtype=np.float64)

print(f"Raw counts matrix: {X_raw.shape}")
print(f"Value range: [{X_raw.min():.0f}, {X_raw.max():.0f}]")
print(f"Sparsity: {(X_raw == 0).mean():.1%}")

# Verify features match
assert list(adata_raw.var_names) == list(
    adata.var_names
), "Feature mismatch between raw counts and harmony lineage files!"
print("\u2705 Features match between raw counts and harmony lineage files.")

del adata_raw  # free memory

# Strategy: Identify genes with shifting 3'UTR isoform proportions along pseudotime

## Motivation

Each gene in this dataset can have multiple 3'UTR isoforms (alternative polyadenylation sites). The total expression of a gene must be _distributed_ across these sites, making this a **compositional / proportions** problem. We want to identify genes whose **relative usage of UTR isoforms** changes along the SVZ neurogenesis trajectory (`dpt_pseudotime`), indicating alternative polyadenylation (APA) regulation during differentiation.

This is distinct from asking whether an individual UTR site's absolute counts change (which is confounded by overall gene expression changes). Instead, we ask: **holding total gene expression constant, does the _fraction_ assigned to each UTR site shift as cells progress from stem cells toward neuroblasts/neurons?**

## How this differs from the two complementary analyses

| Aspect                | Dirichlet Multinomial GLM (`260308-dirichlet_multinomial_analysis.R`) | Graph-smoothed counts Spearman (`260309-..._graphsmoothed.ipynb`) | **Graph-smoothed proportions + entropy (this notebook)**                      |
| --------------------- | --------------------------------------------------------------------- | ----------------------------------------------------------------- | ----------------------------------------------------------------------------- |
| **Question**          | Do UTR isoform proportions shift along pseudotime?                    | Does each UTR's absolute expression change?                       | Do UTR proportions shift? Does UTR diversity change?                          |
| **Batch correction**  | Batch as covariate in GLM                                             | Harmony-corrected graph smoothing                                 | Harmony-corrected graph smoothing                                             |
| **Statistical model** | Parametric (DM likelihood, spline basis, LR test)                     | Non-parametric (Spearman on smoothed counts)                      | Non-parametric (Spearman on smoothed proportions + Shannon entropy)           |
| **Unit of analysis**  | Gene (multivariate: all sites jointly)                                | Individual UTR site                                               | Gene (via site-level proportions + gene-level entropy)                        |
| **Key strength**      | Proper compositional model with formal hypothesis test                | Detects total expression dynamics                                 | Leverages the integrated manifold; entropy captures determinism vs. diversity |

The DM GLM and this analysis ask a similar biological question (APA shifts) but use **completely different statistical frameworks and batch correction strategies**. The DM uses parametric spline models with raw counts and explicit batch covariates; this analysis uses non-parametric rank correlations on graph-smoothed proportions computed from the Harmony-corrected manifold. Concordant hits between the two are high-confidence APA candidates.

## Approach

1. **Graph-smooth the raw counts** using the Harmony-corrected neighbors graph (2 rounds of neighbor-averaging). This denoises expression while implicitly correcting batch effects through the corrected graph topology.

2. **Convert to within-gene proportions**: For each gene with ≥2 UTR sites, divide each site's smoothed count by the gene's total smoothed count per cell. This produces a proportions matrix where, within each gene, values sum to 1 per cell.

3. **Per-site Spearman ρ of proportion vs. pseudotime**: For each UTR site within multi-site genes, correlate its proportion with pseudotime. Summarize at the gene level by taking the maximum |ρ| across sites.

4. **Per-gene Shannon entropy of UTR usage vs. pseudotime**: For each gene, compute the Shannon entropy of its UTR proportions per cell (high entropy = diverse/uniform usage; low entropy = one dominant site). Correlate entropy with pseudotime to detect genes whose UTR determinism changes during differentiation.

5. **Combine and rank** genes by both proportion-shift and entropy statistics.

## Note on p-values

P-values in this analysis are **anti-conservative (inflated)** due to two related sources of non-independence that violate the IID assumption underlying the Spearman test:

1. **Graph smoothing induces spatial autocorrelation.** After 2 rounds of neighbor-averaging, each cell's smoothed expression is a weighted mean over its local neighborhood (~10–30 cells depending on the graph). This means neighboring cells have nearly identical smoothed values, dramatically reducing the effective number of independent observations (effective N ≪ actual N). A standard Spearman p-value computed as if all cells were independent will therefore be far too small.

2. **The neighbors graph is correlated with pseudotime.** The Harmony-corrected graph was built from UTR count data (normalize → log1p → scale → PCA → Harmony), which strongly reflects cellular differentiation state. Cells that are near one another in this graph consequently also tend to have similar pseudotimes. Smoothing over such a graph further sharpens the association between smoothed expression and pseudotime, compounding the inflation.

Note: pseudotime (`dpt_pseudotime`) was computed from a **separate, traditional gene-expression count matrix** — not from this UTR dataset. This means the two inputs to the Spearman test (smoothed proportions and pseudotime) were derived from different primary data sources. However, this does not resolve the anti-conservative issue: the UTR-based graph is still correlated with pseudotime (point 2 above), so the inflation remains.

We report p-values and BH-adjusted p-values for completeness, but **gene selection relies on effect-size thresholds** (|ρ|) rather than p-value cutoffs.


In [ ]:
# Verify neighbors graph
print("Neighbors params:", adata.uns.get("neighbors", {}).get("params", {}))
print("Connectivities shape:", adata.obsp["connectivities"].shape)
print(f"Number of cells: {adata.n_obs}")
print(f"Number of UTR sites (features): {adata.n_vars}")

# Check the var columns
print(f"\nadata.var columns: {list(adata.var.columns)}")
display(adata.var.head(10))

## Build UTR site grouping by gene

We use the `name` column in `adata.var` (ENSEMBL gene ID) to group UTR sites belonging to the same gene. Only genes with ≥2 UTR sites are retained -> single-site genes always have proportion = 1.0 and carry no information about isoform switching.


In [ ]:
# Build gene -> UTR site mapping
var_names = list(adata.var_names)
var_name_to_idx = {v: i for i, v in enumerate(var_names)}

gene_to_sites = defaultdict(list)
for site_name, row in adata.var.iterrows():
    gene_id = row["name"]  # ENSEMBL gene ID
    gene_to_sites[gene_id].append(site_name)

# Filter to multi-site genes only
multi_site_genes = {g: sites for g, sites in gene_to_sites.items() if len(sites) >= 2}

# Build gene_name lookup (ENSEMBL -> symbol)
gene_id_to_name = adata.var.groupby("name")["gene_name"].first().to_dict()

# Summary
n_sites_per_gene = [len(s) for s in multi_site_genes.values()]
print(f"Total genes: {len(gene_to_sites)}")
print(f"Multi-site genes (≥2 UTR sites): {len(multi_site_genes)}")
print(f"Single-site genes (excluded): {len(gene_to_sites) - len(multi_site_genes)}")
print(f"Total UTR sites in multi-site genes: {sum(n_sites_per_gene)}")
print(f"\nDistribution of sites per gene (multi-site only):")
sites_series = pd.Series(n_sites_per_gene)
print(sites_series.describe())
print(f"\nSites per gene value counts:")
print(sites_series.value_counts().sort_index())

## Compute graph-smoothed expression

We row-normalize the Harmony-corrected connectivities matrix and apply 2 rounds of neighbor-averaging to denoise the raw counts while implicitly correcting batch effects through the graph topology.


In [ ]:
# Row-normalize the Harmony-corrected connectivities matrix
conn = adata.obsp["connectivities"].copy()

row_sums = np.array(conn.sum(axis=1)).flatten()
row_sums[row_sums == 0] = 1  # avoid division by zero for isolated cells
norm_conn = diags(1.0 / row_sums) @ conn

# Apply iterative graph smoothing to RAW COUNTS
# (adata.X contains z-scored values from normalize -> log1p -> scale;
#  we smooth the raw counts so that proportions are computed correctly)
n_rounds = 2
X_smooth = X_raw.copy()
for i in range(n_rounds):
    X_smooth = norm_conn @ X_smooth
    print(f"Smoothing round {i + 1}/{n_rounds} complete")

print(f"\nSmoothed expression matrix shape: {X_smooth.shape}")
print(f"Sparsity before smoothing: {(X_raw == 0).mean():.1%}")
print(f"Sparsity after smoothing:  {(X_smooth == 0).mean():.1%}")

## Convert smoothed counts to within-gene proportions

For each multi-site gene, we divide each UTR site's smoothed count by the total gene expression per cell. This produces proportions that sum to 1.0 within each gene for each cell. Cells with zero total expression for a gene get uniform proportions (1/n_sites) as a conservative default.


In [ ]:
# Compute within-gene proportions from smoothed counts
P_smooth = np.zeros_like(X_smooth)

for gene_id, site_names in multi_site_genes.items():
    site_idxs = [var_name_to_idx[s] for s in site_names]
    # Sum across this gene's UTR sites per cell
    gene_total = X_smooth[:, site_idxs].sum(axis=1, keepdims=True)
    # Avoid division by zero: where gene total is 0, set uniform proportions
    zero_mask = gene_total.flatten() == 0
    n_sites = len(site_idxs)
    for j, idx in enumerate(site_idxs):
        P_smooth[:, idx] = np.where(
            zero_mask,
            1.0 / n_sites,
            X_smooth[:, idx] / gene_total.flatten(),
        )

# Sanity check: proportions sum to 1 within each gene
sample_gene = list(multi_site_genes.keys())[0]
sample_sites = multi_site_genes[sample_gene]
sample_idxs = [var_name_to_idx[s] for s in sample_sites]
prop_sums = P_smooth[:, sample_idxs].sum(axis=1)
print(f"Sanity check — gene {sample_gene} ({gene_id_to_name.get(sample_gene, '?')}):")
print(f"  Sites: {sample_sites}")
print(
    f"  Proportion sums: min={prop_sums.min():.6f}, max={prop_sums.max():.6f}, mean={prop_sums.mean():.6f}"
)
print(f"\nProportions matrix shape: {P_smooth.shape}")

# Gene-level Spearman ρ of W_cell and per-cell entropy vs pseudotime

For each multi-site gene we collapse cell-level smoothed proportions to **two** complementary scalar summaries and correlate each with pseudotime:

1. **ρ_W = Spearman(W_c, t_c)** where $W_c = \sum_k r_k\,\pi_k^{(c)}$ — does the _centroid_ of UTR usage shift along pseudotime? Direction matches Δψ; magnitude indicates how monotonically it shifts.

2. **entropy_rho = Spearman(H_c, t_c)** where $H_c = -\sum_k \pi_k^{(c)} \log_2 \pi_k^{(c)} / \log_2 K$ (Shannon entropy normalised to $[0,1]$) — does UTR usage become more _concentrated_ on a single site (entropy*rho < 0) or more \_spread* across sites (entropy_rho > 0)?

These are independent dimensions: a gene can have Δψ ≈ 0 (no centroid shift) but a strong entropy*rho (the \_spread* of usage changes even if the mean position does not). Both are needed for a full description of compositional dynamics.

The previous "best per-site ρ" aggregation suffered from (1) max-of-K selection bias inflating with the number of UTR sites, (2) compositional non-independence across the K constrained per-site correlations, and (3) the loss of a clean gene-level interpretation. Both ρ_W and entropy_rho are statistically independent of the DirMult model fit (graph smoothing operates on the Harmony embedding, not on the spline basis), making them genuine triangulating metrics for the model-derived Δψ rather than tautologies of the same fit.

See `260504-effect_size_metrics_methods.md` §4 for the full statistical rationale.


In [ ]:
pseudotime = adata.obs["dpt_pseudotime"].values
all_utr_sizes = adata.var["utr_size"].values.astype(float)

# Pre-compute per-gene metadata once (site indices, ranks, K).
gene_meta = []
for gene_id, site_names in multi_site_genes.items():
    site_idxs = np.fromiter(
        (var_name_to_idx[s] for s in site_names),
        dtype=np.int64,
        count=len(site_names),
    )
    K = len(site_idxs)
    raw_ranks = rankdata(all_utr_sizes[site_idxs], method="average")
    ranks = (raw_ranks - 1) / (K - 1)  # shortest -> 0, longest -> 1
    gene_meta.append((gene_id, gene_id_to_name.get(gene_id, ""), site_idxs, K, ranks))

# Inner loop computes BOTH metrics from the same per-cell proportions:
#   ρ_W         = Spearman(W_c, t)        (centroid shift along pseudotime)
#   entropy_rho = Spearman(H_c, t)        (concentration vs spread of UTR usage)
#
# Speed: column-slice K columns first (small intermediate), then row-mask.
# The previous form `P_smooth[expressing][:, site_idxs]` materialised a
# (n_expr × n_features) ~3 GB array per gene; this version touches only
# (n_cells × K) ~350 kB.
results = []
for i, (gene_id, gene_name, site_idxs, K, ranks) in enumerate(gene_meta):
    if (i + 1) % 500 == 0 or i == 0:
        print(f"Processing gene {i + 1}/{len(gene_meta)}...")

    sub = X_smooth[:, site_idxs]  # (n_cells, K)
    gene_total = sub.sum(axis=1)  # (n_cells,)
    expressing = gene_total > 0
    n_expr = int(expressing.sum())
    if n_expr < 30:
        results.append((gene_id, gene_name, K, n_expr, np.nan, np.nan, np.nan, np.nan))
        continue

    # Per-cell proportions for expressing cells only
    sub_expr = sub[expressing]  # (n_expr, K)
    total_expr = gene_total[expressing]  # (n_expr,)
    props = sub_expr / total_expr[:, None]  # (n_expr, K), rows sum to 1

    # 1. W_cell = Σ r_k · π_k(c)  — rank-weighted UTR-length centroid per cell
    W_cell = props @ ranks

    # 2. Shannon entropy per cell, normalised to [0, 1]: 0 = single site, 1 = uniform
    with np.errstate(divide="ignore", invalid="ignore"):
        log_p = np.where(props > 0, np.log2(props), 0.0)
    H_per_cell = -np.sum(props * log_p, axis=1)
    if K > 1:
        H_per_cell = H_per_cell / np.log2(K)

    pt_expr = pseudotime[expressing]

    # ρ_W
    if W_cell.std() == 0:
        rho_W, pval_W = 0.0, 1.0
    else:
        rho_W, pval_W = spearmanr(pt_expr, W_cell)

    # entropy_rho — positive: UTR usage spreads along pseudotime; negative: concentrates
    if H_per_cell.std() == 0:
        entropy_rho, pval_H = 0.0, 1.0
    else:
        entropy_rho, pval_H = spearmanr(pt_expr, H_per_cell)

    results.append((gene_id, gene_name, K, n_expr, rho_W, pval_W, entropy_rho, pval_H))

df_gene = pd.DataFrame(
    results,
    columns=[
        "gene_id",
        "gene_name",
        "n_sites",
        "n_cells",
        "rho_W",
        "pvalue_W",
        "entropy_rho",
        "pvalue_H",
    ],
)


def _bh(pvals):
    out = np.full(len(pvals), np.nan)
    valid = ~np.isnan(pvals)
    if valid.sum() > 0:
        _, padj_valid, _, _ = multipletests(pvals[valid], method="fdr_bh")
        out[valid] = padj_valid
    return out


df_gene["padj_BH_W"] = _bh(df_gene["pvalue_W"].values)
df_gene["padj_BH_H"] = _bh(df_gene["pvalue_H"].values)
df_gene["abs_rho_W"] = df_gene["rho_W"].abs()
df_gene["abs_entropy_rho"] = df_gene["entropy_rho"].abs()

df_gene = df_gene.sort_values(
    "abs_rho_W", ascending=False, na_position="last"
).reset_index(drop=True)

print("\n--- ρ_W (rank-weighted shift) summary ---")
print(f"Total multi-site genes tested: {len(df_gene)}")
print(f"Genes with valid ρ_W: {df_gene['rho_W'].notna().sum()}")
print(f"Genes with |ρ_W| > 0.3: {(df_gene['abs_rho_W'] > 0.3).sum()}")
print(
    f"Genes with |ρ_W| > 0.3 & padj_BH_W < 0.05: {((df_gene['abs_rho_W'] > 0.3) & (df_gene['padj_BH_W'] < 0.05)).sum()}"
)

print("\n--- entropy_rho (concentration vs spread) summary ---")
print(f"Genes with valid entropy_rho: {df_gene['entropy_rho'].notna().sum()}")
print(
    f"  entropy_rho > 0  (UTR usage SPREADING along pseudotime):    {(df_gene['entropy_rho'] > 0).sum():,}"
)
print(
    f"  entropy_rho < 0  (UTR usage CONCENTRATING along pseudotime): {(df_gene['entropy_rho'] < 0).sum():,}"
)
print(f"  |entropy_rho| > 0.3: {(df_gene['abs_entropy_rho'] > 0.3).sum()}")
print(
    f"  |entropy_rho| > 0.3 & padj_BH_H < 0.05: {((df_gene['abs_entropy_rho'] > 0.3) & (df_gene['padj_BH_H'] < 0.05)).sum()}"
)

df_gene.head(20).style.format(
    {
        "rho_W": "{:.3f}",
        "abs_rho_W": "{:.3f}",
        "pvalue_W": "{:.2e}",
        "padj_BH_W": "{:.2e}",
        "entropy_rho": "{:.3f}",
        "abs_entropy_rho": "{:.3f}",
        "pvalue_H": "{:.2e}",
        "padj_BH_H": "{:.2e}",
    }
)

## Distribution of ρ_W


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].hist(
    df_gene["rho_W"].dropna(),
    bins=60,
    color="#2980b9",
    edgecolor="black",
    linewidth=0.3,
)
axes[0].axvline(0, color="black", linewidth=0.8, linestyle="--")
axes[0].set_xlabel("ρ_W (gene-level Spearman)")
axes[0].set_ylabel("Genes")
axes[0].set_title("Signed ρ_W distribution")

sig_mask = df_gene["padj_BH_W"] < 0.05
axes[1].hist(
    df_gene.loc[~sig_mask, "abs_rho_W"].dropna(),
    bins=40,
    alpha=0.5,
    label="padj_BH_W ≥ 0.05",
    color="lightgrey",
    edgecolor="black",
    linewidth=0.3,
)
axes[1].hist(
    df_gene.loc[sig_mask, "abs_rho_W"].dropna(),
    bins=40,
    alpha=0.7,
    label="padj_BH_W < 0.05",
    color="firebrick",
    edgecolor="black",
    linewidth=0.3,
)
axes[1].set_xlabel("|ρ_W|")
axes[1].set_ylabel("Genes")
axes[1].set_title("|ρ_W| stratified by significance")
axes[1].legend()
plt.tight_layout()
plt.show()

## Distribution of entropy_rho

Symmetric to the ρ*W panel above, but for the orthogonal \_concentration vs spreading* axis. Negative `entropy_rho` = UTR usage _concentrating_ on a single site along pseudotime; positive = _spreading_.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].hist(
    df_gene["entropy_rho"].dropna(),
    bins=60,
    color="mediumseagreen",
    edgecolor="black",
    linewidth=0.3,
)
axes[0].axvline(0, color="black", linewidth=0.8, linestyle="--")
axes[0].set_xlabel(
    "entropy_rho (gene-level Spearman of normalised Shannon entropy vs pseudotime)"
)
axes[0].set_ylabel("Genes")
axes[0].set_title("Signed entropy_rho distribution")

sig_mask_H = df_gene["padj_BH_H"] < 0.05
axes[1].hist(
    df_gene.loc[~sig_mask_H, "abs_entropy_rho"].dropna(),
    bins=40,
    alpha=0.5,
    label="padj_BH_H ≥ 0.05",
    color="lightgrey",
    edgecolor="black",
    linewidth=0.3,
)
axes[1].hist(
    df_gene.loc[sig_mask_H, "abs_entropy_rho"].dropna(),
    bins=40,
    alpha=0.7,
    label="padj_BH_H < 0.05",
    color="seagreen",
    edgecolor="black",
    linewidth=0.3,
)
axes[1].set_xlabel("|entropy_rho|")
axes[1].set_ylabel("Genes")
axes[1].set_title("|entropy_rho| stratified by significance")
axes[1].legend()
plt.tight_layout()
plt.show()

## Joint view: ρ_W vs entropy_rho — directional shift vs concentration/spread

Each point is a multi-site gene. x-axis is the centroid shift (`rho_W`), y-axis is the concentration/spread shift (`entropy_rho`). The two metrics are orthogonal: a gene can shift its UTR centroid without changing entropy, or change entropy without shifting the centroid. Significance threshold uses |ρ| > 0.3 (matches the older notebooks' convention).


In [ ]:
rho_thresh = 0.3
sig_W_mask = df_gene["abs_rho_W"] > rho_thresh
sig_H_mask = df_gene["abs_entropy_rho"] > rho_thresh
both_sig = sig_W_mask & sig_H_mask
only_W = sig_W_mask & ~sig_H_mask
only_H = ~sig_W_mask & sig_H_mask

colors = np.full(len(df_gene), "lightgrey", dtype=object)
colors[only_W.values] = "orange"
colors[only_H.values] = "mediumseagreen"
colors[both_sig.values] = "firebrick"

fig, ax = plt.subplots(figsize=(8.5, 7.5))
ax.scatter(
    df_gene["rho_W"],
    df_gene["entropy_rho"],
    c=colors,
    s=8,
    alpha=0.6,
    edgecolors="none",
)

# Annotate top "both significant" genes by combined |rho_W| * |entropy_rho|
both_df = df_gene.loc[both_sig].copy()
both_df["combined"] = both_df["abs_rho_W"] * both_df["abs_entropy_rho"]
top_both = both_df.sort_values("combined", ascending=False).head(12)
for _, row in top_both.iterrows():
    ax.annotate(
        row["gene_name"],
        (row["rho_W"], row["entropy_rho"]),
        fontsize=7,
        alpha=0.85,
        ha="left",
        va="bottom",
    )

for v in (0,):
    ax.axhline(v, color="black", linewidth=0.4)
    ax.axvline(v, color="black", linewidth=0.4)
for v in (-rho_thresh, rho_thresh):
    ax.axhline(v, color="grey", linestyle="--", linewidth=0.5, alpha=0.5)
    ax.axvline(v, color="grey", linestyle="--", linewidth=0.5, alpha=0.5)

ax.set_xlabel("ρ_W (rank-weighted centroid shift vs pseudotime)")
ax.set_ylabel("entropy_rho (UTR diversity shift vs pseudotime)")
ax.set_title(
    "Gene-level orthogonal axes: directional shift (x) vs concentration/spread (y)"
)

legend_elements = [
    Line2D(
        [0],
        [0],
        marker="o",
        color="w",
        markerfacecolor="firebrick",
        markersize=7,
        label=f"Both |ρ| > {rho_thresh}  (n={int(both_sig.sum())})",
    ),
    Line2D(
        [0],
        [0],
        marker="o",
        color="w",
        markerfacecolor="orange",
        markersize=7,
        label=f"ρ_W only  (n={int(only_W.sum())})",
    ),
    Line2D(
        [0],
        [0],
        marker="o",
        color="w",
        markerfacecolor="mediumseagreen",
        markersize=7,
        label=f"entropy only  (n={int(only_H.sum())})",
    ),
    Line2D(
        [0],
        [0],
        marker="o",
        color="w",
        markerfacecolor="lightgrey",
        markersize=7,
        label="neither",
    ),
]
ax.legend(handles=legend_elements, loc="upper center", fontsize=9, framealpha=0.9)

plt.tight_layout()
plt.show()

## Entropy trajectories along pseudotime for top genes

For the top genes by |entropy_rho|, plot binned mean ± SEM of per-cell normalised Shannon entropy across pseudotime. Decreasing curves = UTR usage concentrating on a single site as differentiation progresses; increasing curves = UTR usage spreading across sites. Per-cell entropy is recomputed on the fly from `X_smooth` to avoid materialising an `(n_cells × n_genes)` entropy matrix.


In [ ]:
def _entropy_trajectory_for_gene(
    gene_id,
    n_bins,
    X_smooth,
    pseudotime,
    multi_site_genes,
    var_name_to_idx,
    bin_indices,
):
    """Recompute per-cell entropy for one gene and bin along pseudotime.

    Returns (bin_means, bin_sems) of length n_bins, with NaN for empty bins.
    """
    site_names = multi_site_genes[gene_id]
    site_idxs = [var_name_to_idx[s] for s in site_names]
    K = len(site_idxs)

    sub = X_smooth[:, site_idxs]
    gene_total = sub.sum(axis=1)
    expressing = gene_total > 0

    H = np.full(X_smooth.shape[0], np.nan)
    if expressing.sum() > 0 and K > 1:
        props = sub[expressing] / gene_total[expressing, None]
        with np.errstate(divide="ignore", invalid="ignore"):
            log_p = np.where(props > 0, np.log2(props), 0.0)
        H_e = -np.sum(props * log_p, axis=1) / np.log2(K)
        H[expressing] = H_e

    bin_means = np.full(n_bins, np.nan)
    bin_sems = np.zeros(n_bins)
    for b in range(n_bins):
        sel = (bin_indices == b) & ~np.isnan(H)
        n_sel = int(sel.sum())
        if n_sel > 0:
            vals = H[sel]
            bin_means[b] = vals.mean()
            if n_sel > 1:
                bin_sems[b] = vals.std() / np.sqrt(n_sel)
    return bin_means, bin_sems


def plot_entropy_panels(gene_ids, title, color, n_bins=40, ncols=3):
    bin_edges = np.linspace(0, 1, n_bins + 1)
    bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
    bin_indices = np.digitize(pseudotime, bin_edges) - 1
    bin_indices = np.clip(bin_indices, 0, n_bins - 1)

    n = len(gene_ids)
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(
        nrows, ncols, figsize=(4.5 * ncols, 3.2 * nrows), squeeze=False
    )
    rho_lookup = df_gene.set_index("gene_id")["entropy_rho"]

    for i, gene_id in enumerate(gene_ids):
        ax = axes[i // ncols, i % ncols]
        bin_means, bin_sems = _entropy_trajectory_for_gene(
            gene_id,
            n_bins,
            X_smooth,
            pseudotime,
            multi_site_genes,
            var_name_to_idx,
            bin_indices,
        )
        ax.fill_between(
            bin_centers,
            bin_means - bin_sems,
            bin_means + bin_sems,
            alpha=0.3,
            color=color,
        )
        ax.plot(bin_centers, bin_means, color=color, linewidth=1.5)
        gname = gene_id_to_name.get(gene_id, gene_id)
        rho = rho_lookup.get(gene_id, np.nan)
        ax.set_title(f"{gname}\nentropy_rho = {rho:.3f}", fontsize=10)
        ax.set_xlabel("Pseudotime")
        ax.set_ylabel("Normalised Shannon entropy")
        ax.set_xlim(0, 1)
        ax.set_ylim(0, 1)

    for i in range(n, nrows * ncols):
        axes[i // ncols, i % ncols].set_visible(False)

    fig.suptitle(title, fontsize=12, y=1.02)
    plt.tight_layout()
    plt.show()


# Top 9 genes with the most NEGATIVE entropy_rho (UTR usage concentrating on a single site)
top_concentrating = (
    df_gene.dropna(subset=["entropy_rho"])
    .sort_values("entropy_rho", ascending=True)
    .head(9)["gene_id"]
    .tolist()
)
if top_concentrating:
    plot_entropy_panels(
        top_concentrating,
        title="Top genes CONCENTRATING UTR usage along pseudotime (entropy_rho << 0)",
        color="steelblue",
    )

# Top 9 genes with the most POSITIVE entropy_rho (UTR usage spreading across sites)
top_spreading = (
    df_gene.dropna(subset=["entropy_rho"])
    .sort_values("entropy_rho", ascending=False)
    .head(9)["gene_id"]
    .tolist()
)
if top_spreading:
    plot_entropy_panels(
        top_spreading,
        title="Top genes SPREADING UTR usage along pseudotime (entropy_rho >> 0)",
        color="firebrick",
    )

## Save results


In [ ]:
outpath = os.path.join(STOREPATH, "utr_pseudotime_W_spearman_per_gene.csv")
df_gene.to_csv(outpath, index=False)
print(f"Saved gene-level ρ_W table to {outpath}")
print(f"  rows: {len(df_gene)}")
print(f"  columns: {list(df_gene.columns)}")